In [ ]:
### query vector_store


## 
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
import pandas as pd

# 1. Point Chroma directly to your local Ollama instance running Qwen3
ollama_ef = OllamaEmbeddingFunction(
    url="http://localhost:11434",
    model_name="qwen3-embedding:latest"
)

# 2. Connect to your database tracking directory
client = chromadb.PersistentClient(path="./chroma_tariff_db")


# 3. Pull the collection, explicitly passing the Ollama embedding function
collection = client.get_collection(
    name="tariff_sections_v2", 
    embedding_function=ollama_ef
)

# 4. Query using natural text strings
# Chroma will handle generating the 4096-dim vector via Ollama under the hood!
query_results = collection.query(
    query_texts=["What dues are available in the document?"],
    n_results=5
)

# 5. Format cleanly into a pandas DataFrame (unwrapping the batch index 0)
df = pd.DataFrame({
    "id": query_results["ids"][0],
    "document": query_results["documents"][0],
    "distance": query_results["distances"][0],
    "metadata": query_results["metadatas"][0] 
})

# 6. Unpack individual metadata dictionaries into separate columns
df = pd.concat([df.drop(['metadata'], axis=1), df['metadata'].apply(pd.Series)], axis=1)

print(df)

In [13]:
# 6. Unpack metadata into columns when the query result still has it nested
if "metadata" in df.columns:
    metadata_df = df.pop("metadata").apply(pd.Series)
    df = pd.concat([df, metadata_df], axis=1)

# 7. Extract only headings that are actually present in each chunk
heading_pattern = r"(?m)^\s*(\d+(?:\.\d+)+)\s+([A-Z][A-Z /&'\\-]{3,})\s*$"
headings = df["document"].str.extract(heading_pattern)
df["heading_in_chunk"] = headings[0].fillna("") + " " + headings[1].fillna("")
df["heading_in_chunk"] = df["heading_in_chunk"].str.strip()
df["heading_in_chunk"] = df["heading_in_chunk"].replace("", "HEADING NOT IN CHUNK")

# v2 metadata is legacy carry-forward metadata and must not be treated as authoritative.
df["stored_section"] = df.get("tariff_section", "NO STORED SECTION")
df["document_preview"] = (
    df["document"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.slice(0, 180)
)

section_view = df[
    ["heading_in_chunk", "stored_section", "page", "distance", "document_preview"]
].sort_values(["heading_in_chunk", "distance"], na_position="last")

print("Actual headings found inside returned chunks:")
print(section_view[["heading_in_chunk", "page"]].drop_duplicates().to_string(index=False))
print("\nNote: 'stored_section' is legacy v2 metadata and may be incorrect.")
print("\nRetrieved chunks:")
display(section_view)

Actual headings found inside returned chunks:
    heading_in_chunk  page
8.1 CARGO DUES ORDER    23
HEADING NOT IN CHUNK     1
HEADING NOT IN CHUNK    23
HEADING NOT IN CHUNK    25

Note: 'stored_section' is legacy v2 metadata and may be incorrect.

Retrieved chunks:


,heading_in_chunk,stored_section,page,distance,document_preview
0,8.1 CARGO DUES ORDER,8.1 CARGO DUES ORDER,23,0.815764,SELS WITHIN AND OUTSIDE THE PORT Per trip……………...
1,HEADING NOT IN CHUNK,1.1 LIGHT DUES,1,0.844299,syncrolift dues 6.7 Slipway 6.8 Wharf cranes 6...
2,HEADING NOT IN CHUNK,8.1 CARGO DUES ORDER,23,0.869112,"refilling, as well as new MT ISO containers wi..."
3,HEADING NOT IN CHUNK,8.1 CARGO DUES ORDER,23,0.906512,cable from the invoicing date. The original or...
4,HEADING NOT IN CHUNK,8.1.2 TIMING OF DOCUMENTATION,25,0.907098,4. Manifests in respects of empties must be su...
